# 02 — Apache Iceberg com PySpark

Este notebook demonstra o ciclo completo de operações em uma tabela **Apache Iceberg** usando o mesmo modelo estrela criado em `00_modelagem_dados.ipynb`:

1. Criar SparkSession com Iceberg-Spark-Runtime.
2. Criar e popular as 3 tabelas (`dim_type`, `dim_generation`, `fact_pokemon`).
3. **INSERT** de novos Pokémons (Gen 7) via SQL.
4. **UPDATE** SQL — rebalancear stats de lendários.
5. **DELETE** SQL — remover Pokémons fracos.
6. **MERGE INTO** — upsert.
7. **Schema Evolution** — `ALTER TABLE ADD COLUMN`.
8. **Partition Evolution** — `ADD PARTITION FIELD` (recurso **exclusivo** do Iceberg).
9. **Time Travel** — `VERSION AS OF` e `TIMESTAMP AS OF`.
10. **Metadata tables** — `.snapshots`, `.history`, `.files`.
11. **Compaction** — `rewrite_data_files`.

> **Pré-requisito**: rodar primeiro `00_modelagem_dados.ipynb` para gerar `/workspace/data/gold/`.

> **Diferenças vs Delta**: Iceberg usa **catálogos lógicos** (`local.pokedex.fact_pokemon`) e SQL puro para a maior parte das operações. O log é em arquivos JSON de **metadata** (snapshots, manifests), enquanto Delta usa `_delta_log/`.

## 1. SparkSession com Iceberg

Carregamos o JAR `iceberg-spark-runtime-3.5_2.12:1.6.1` via `spark.jars.packages` (Maven). Na primeira execução, o JAR é baixado para o cache Ivy (`~/.ivy2/`) — pode demorar ~30s; nas execuções seguintes é instantâneo (volume `ivy-cache` do docker-compose).

Catálogo `local`: tipo `hadoop`, warehouse em `/workspace/data/iceberg/warehouse`. Não precisa de Hive Metastore — perfeito para demo local.

In [ ]:
from pyspark.sql import SparkSession, functions as F

ICEBERG_WAREHOUSE = "/workspace/data/iceberg/warehouse"
GOLD_DIR          = "/workspace/data/gold"

spark = (
    SparkSession.builder
    .appName("IcebergPokemon")
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", ICEBERG_WAREHOUSE)
    .config("spark.sql.defaultCatalog", "local")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Iceberg warehouse:", ICEBERG_WAREHOUSE)

## 2. Criar e popular tabelas Iceberg

Cria o namespace `local.pokedex` e as 3 tabelas. `fact_pokemon` é particionada por `generation_id`.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.pokedex")

# Drop se existirem (idempotência: notebook pode ser re-executado)
for t in ("fact_pokemon", "dim_type", "dim_generation"):
    spark.sql(f"DROP TABLE IF EXISTS local.pokedex.{t} PURGE")

spark.sql("""
CREATE TABLE local.pokedex.dim_type (
    type_id   INT,
    type_name STRING
) USING iceberg
""")

spark.sql("""
CREATE TABLE local.pokedex.dim_generation (
    generation_id   INT,
    generation_name STRING,
    era             STRING
) USING iceberg
""")

spark.sql("""
CREATE TABLE local.pokedex.fact_pokemon (
    pokemon_id    INT,
    name          STRING,
    type_1_id     INT,
    type_2_id     INT,
    hp            INT,
    attack        INT,
    defense       INT,
    sp_atk        INT,
    sp_def        INT,
    speed         INT,
    total         INT,
    is_legendary  BOOLEAN,
    generation_id INT
) USING iceberg
PARTITIONED BY (generation_id)
""")

spark.sql("SHOW TABLES IN local.pokedex").show()

In [ ]:
# Carregar gold do Parquet e popular as tabelas Iceberg
spark.read.parquet(f"{GOLD_DIR}/dim_type")       .writeTo("local.pokedex.dim_type")       .append()
spark.read.parquet(f"{GOLD_DIR}/dim_generation") .writeTo("local.pokedex.dim_generation") .append()
spark.read.parquet(f"{GOLD_DIR}/fact_pokemon")   .writeTo("local.pokedex.fact_pokemon")   .append()

spark.sql("SELECT COUNT(*) AS n FROM local.pokedex.fact_pokemon").show()
spark.sql("SELECT * FROM local.pokedex.fact_pokemon LIMIT 5").show()

In [ ]:
spark.sql("DESCRIBE EXTENDED local.pokedex.fact_pokemon").show(truncate=False, n=50)

## 3. INSERT — Pokémons fictícios da Gen 7

Iceberg usa SQL puro: `INSERT INTO`. A nova partição (`generation_id=7`) é criada automaticamente.

In [ ]:
# Adicionar Gen 7 na dim
spark.sql("""
INSERT INTO local.pokedex.dim_generation VALUES
    (7, 'Alola', 'Modern')
""")
spark.sql("SELECT * FROM local.pokedex.dim_generation ORDER BY generation_id").show()

In [ ]:
# Buscar IDs dos tipos para construir os INSERTs
type_lookup = {
    row["type_name"]: row["type_id"]
    for row in spark.sql("SELECT type_id, type_name FROM local.pokedex.dim_type").collect()
}
next_id = spark.sql("SELECT MAX(pokemon_id) AS m FROM local.pokedex.fact_pokemon").first()["m"] + 1
print("next_id =", next_id)

spark.sql(f"""
INSERT INTO local.pokedex.fact_pokemon VALUES
    ({next_id},     'Decidueye',  {type_lookup['Grass']}, {type_lookup['Ghost']},  78,107,75,100,100,70,530, false, 7),
    ({next_id + 1}, 'Incineroar', {type_lookup['Fire']},  {type_lookup['Dark']},   95,115,90,80,90,60,530,  false, 7),
    ({next_id + 2}, 'Primarina',  {type_lookup['Water']}, {type_lookup['Fairy']},  80,74,74,126,116,60,530, false, 7)
""")

spark.sql("SELECT * FROM local.pokedex.fact_pokemon WHERE generation_id = 7").show()

## 4. UPDATE — rebalancear lendários (-10 attack)

In [ ]:
before = spark.sql("""
    SELECT AVG(attack) AS avg_attack
    FROM local.pokedex.fact_pokemon
    WHERE is_legendary = true
""").first()["avg_attack"]
print(f"Avg attack lendários ANTES: {before:.2f}")

spark.sql("""
    UPDATE local.pokedex.fact_pokemon
    SET attack = attack - 10
    WHERE is_legendary = true
""")

after = spark.sql("""
    SELECT AVG(attack) AS avg_attack
    FROM local.pokedex.fact_pokemon
    WHERE is_legendary = true
""").first()["avg_attack"]
print(f"Avg attack lendários DEPOIS: {after:.2f}")

## 5. DELETE — remover Pokémons com `total < 250`

In [ ]:
before_n = spark.sql("SELECT COUNT(*) AS n FROM local.pokedex.fact_pokemon").first()["n"]
to_del   = spark.sql("SELECT COUNT(*) AS n FROM local.pokedex.fact_pokemon WHERE total < 250").first()["n"]
print(f"Linhas antes: {before_n} | Serão deletadas: {to_del}")

spark.sql("DELETE FROM local.pokedex.fact_pokemon WHERE total < 250")

after_n = spark.sql("SELECT COUNT(*) AS n FROM local.pokedex.fact_pokemon").first()["n"]
print(f"Linhas depois: {after_n}")

## 6. MERGE INTO — upsert (correções + novos Pokémons)

Iceberg suporta `MERGE INTO` com sintaxe SQL padrão. Vamos criar uma fonte temporária com correções de Bulbasaur, Charmander e um novo Pokémon (Mimikyu).

In [ ]:
max_id = spark.sql("SELECT MAX(pokemon_id) AS m FROM local.pokedex.fact_pokemon").first()["m"]
new_id = max_id + 1

updates = spark.createDataFrame(
    [
        (1,      "Bulbasaur", type_lookup["Grass"], type_lookup["Poison"], 45, 55, 49, 65, 65, 45, 318, False, 1),
        (4,      "Charmander", type_lookup["Fire"], None,                  39, 58, 43, 60, 50, 65, 309, False, 1),
        (new_id, "Mimikyu",   type_lookup["Ghost"], type_lookup["Fairy"],  55, 90, 80, 50, 105, 96, 476, False, 7),
    ],
    schema=("pokemon_id INT, name STRING, type_1_id INT, type_2_id INT, "
            "hp INT, attack INT, defense INT, sp_atk INT, sp_def INT, speed INT, "
            "total INT, is_legendary BOOLEAN, generation_id INT"),
)
updates.createOrReplaceTempView("updates_src")
updates.show()

In [ ]:
spark.sql("""
MERGE INTO local.pokedex.fact_pokemon t
USING updates_src s
ON t.pokemon_id = s.pokemon_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

spark.sql("""
SELECT pokemon_id, name, attack, generation_id
FROM local.pokedex.fact_pokemon
WHERE pokemon_id IN (1, 4) OR name = 'Mimikyu'
ORDER BY pokemon_id
""").show()

## 7. Schema Evolution — `ALTER TABLE ADD COLUMN`

Iceberg evoluí o schema sem reescrever os dados (é apenas uma mudança de metadata).

In [ ]:
spark.sql("""
ALTER TABLE local.pokedex.fact_pokemon
ADD COLUMN mega_evolution BOOLEAN COMMENT 'Pokémon possui mega-evolução'
""")

# Inserir um Mega Charizard X com a coluna nova preenchida
mega_id = spark.sql("SELECT MAX(pokemon_id)+1 AS m FROM local.pokedex.fact_pokemon").first()["m"]
spark.sql(f"""
INSERT INTO local.pokedex.fact_pokemon VALUES
    ({mega_id}, 'Mega Charizard X',
     {type_lookup['Fire']}, {type_lookup['Dragon']},
     78, 130, 111, 130, 85, 100, 634, false, 6, true)
""")

spark.sql("""
SELECT pokemon_id, name, mega_evolution
FROM local.pokedex.fact_pokemon
WHERE mega_evolution = true
""").show()

## 8. Partition Evolution — recurso **exclusivo** do Iceberg

Aqui está a **maior diferença** vs Delta Lake: o Iceberg permite **alterar o esquema de particionamento** de uma tabela existente sem reescrever dados antigos. Os dados antigos continuam particionados pelo esquema antigo; novos dados usam o novo esquema. As queries continuam funcionando por causa do **hidden partitioning**.

Vamos adicionar `is_legendary` como coluna de partição (passa a ser particionada por `generation_id` E `is_legendary`).

In [ ]:
spark.sql("""
ALTER TABLE local.pokedex.fact_pokemon
ADD PARTITION FIELD is_legendary
""")

# Inserir nova linha — será gravada na nova partition spec
lugia_id = spark.sql("SELECT MAX(pokemon_id)+1 AS m FROM local.pokedex.fact_pokemon").first()["m"]
spark.sql(f"""
INSERT INTO local.pokedex.fact_pokemon VALUES
    ({lugia_id}, 'Lugia',
     {type_lookup['Psychic']}, {type_lookup['Flying']},
     106, 80, 130, 90, 154, 110, 670, true, 2, false)
""")

# Mostrar a evolução de partições
spark.sql("SELECT * FROM local.pokedex.fact_pokemon.partitions LIMIT 20").show(truncate=False)

## 9. Time Travel

Cada operação de escrita cria um **snapshot** novo. Podemos consultar:
- por `VERSION AS OF <snapshot_id>`
- por `TIMESTAMP AS OF '<timestamp>'`

In [ ]:
# Listar snapshots
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at, operation, summary
    FROM local.pokedex.fact_pokemon.snapshots
    ORDER BY committed_at
""").collect()

print(f"Total de snapshots: {len(snapshots)}")
for i, s in enumerate(snapshots):
    print(f"  v{i}: {s['snapshot_id']} | {s['committed_at']} | {s['operation']}")

In [ ]:
# Time travel: ler o primeiro snapshot (write inicial — antes dos UPDATEs/DELETEs)
first_snap_id = snapshots[0]["snapshot_id"]

v0_count = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM local.pokedex.fact_pokemon VERSION AS OF {first_snap_id}
""").first()["n"]

current_count = spark.sql("SELECT COUNT(*) AS n FROM local.pokedex.fact_pokemon").first()["n"]

print(f"Linhas no snapshot v0: {v0_count}")
print(f"Linhas agora:          {current_count}")

# Comparar attack médio dos lendários antes/depois do UPDATE
v0_legend = spark.sql(f"""
    SELECT AVG(attack) AS a
    FROM local.pokedex.fact_pokemon VERSION AS OF {first_snap_id}
    WHERE is_legendary = true
""").first()["a"]
now_legend = spark.sql("""
    SELECT AVG(attack) AS a
    FROM local.pokedex.fact_pokemon
    WHERE is_legendary = true
""").first()["a"]
print(f"Avg attack lendários — v0: {v0_legend:.2f} | atual: {now_legend:.2f}")

In [ ]:
# Time travel por TIMESTAMP
ts_v0 = snapshots[0]["committed_at"]
print("Lendo @", ts_v0)
spark.sql(f"""
    SELECT pokemon_id, name, attack
    FROM local.pokedex.fact_pokemon TIMESTAMP AS OF '{ts_v0}'
    WHERE pokemon_id <= 5
    ORDER BY pokemon_id
""").show()

## 10. Metadata tables — auditoria

Iceberg expõe metadata como tabelas SQL: `.snapshots`, `.history`, `.files`, `.manifests`, `.partitions`, `.refs`.

In [ ]:
print("=== HISTORY ===")
spark.sql("SELECT * FROM local.pokedex.fact_pokemon.history").show(truncate=False)

print("=== SNAPSHOTS (resumo) ===")
spark.sql("""
    SELECT snapshot_id, parent_id, operation, summary['added-records'] AS added, summary['deleted-records'] AS deleted
    FROM local.pokedex.fact_pokemon.snapshots
    ORDER BY committed_at
""").show(truncate=False)

print("=== FILES (top 5) ===")
spark.sql("""
    SELECT file_path, partition, record_count, file_size_in_bytes
    FROM local.pokedex.fact_pokemon.files
    LIMIT 5
""").show(truncate=False)

## 11. Compaction — `rewrite_data_files`

Iceberg expõe procedimentos via `CALL`. `rewrite_data_files` compacta arquivos pequenos.

In [ ]:
spark.sql("""
CALL local.system.rewrite_data_files(
    table => 'pokedex.fact_pokemon',
    options => map('min-input-files', '2', 'target-file-size-bytes', '5242880')
)
""").show(truncate=False)

In [ ]:
spark.stop()

## Comparação rápida — Delta Lake × Apache Iceberg

| Recurso | Delta Lake | Apache Iceberg |
|---|---|---|
| Origem | Databricks | Netflix |
| Formato dos dados | Parquet | Parquet, ORC, Avro |
| Metadata | `_delta_log/` (JSON + checkpoint Parquet) | `metadata/` (snapshots + manifests JSON/Avro) |
| API principal | DataFrame (`DeltaTable`) + SQL | **SQL puro** + DataFrame |
| Time travel | `versionAsOf` / `timestampAsOf` | `VERSION AS OF` / `TIMESTAMP AS OF` |
| Schema evolution | Sim (com `mergeSchema`) | Sim (`ALTER TABLE`) |
| **Partition evolution** | ❌ Não | ✅ **Sim** (`ADD PARTITION FIELD`) |
| Hidden partitioning | ❌ | ✅ |
| Compaction | `OPTIMIZE` | `CALL rewrite_data_files` |
| GC de arquivos | `VACUUM` | `CALL expire_snapshots` + `remove_orphan_files` |
| Catálogos | Hive, Unity, AWS Glue | Hive, Hadoop, Glue, Nessie, REST, JDBC |
| Ecossistema | Forte com Spark/Databricks | Forte multi-engine (Spark, Flink, Trino, Presto, Snowflake, Dremio) |

**Quando escolher cada um?**
- **Delta Lake**: stack 100% Spark/Databricks, ou se quiser a integração mais polida com PySpark.
- **Apache Iceberg**: stack multi-engine (Spark + Trino + Flink, etc.), necessidade de partition evolution, ou política de catálogo aberto.